In [2]:
from sqlalchemy import create_engine
import pandas as pd

DB_USER = "postgres"
DB_PASSWORD = "mysecretpassword"
DB_HOST = "localhost"
DB_PORT = "5432"
DB_NAME = "olist"

engine = create_engine(f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}")

# اختبار الاتصال
test = pd.read_sql("SELECT 1", engine)
print(test)
print("✅ الاتصال بقاعدة البيانات نجح!")

   ?column?
0         1
✅ الاتصال بقاعدة البيانات نجح!


In [3]:
df_orders = pd.read_sql("SELECT * FROM orders", engine)
df_customers = pd.read_sql("SELECT * FROM customers", engine)
df_order_items = pd.read_sql("SELECT * FROM order_items", engine)
df_payments = pd.read_sql("SELECT * FROM order_payments", engine)
df_reviews = pd.read_sql("SELECT * FROM order_reviews", engine)
df_products = pd.read_sql("SELECT * FROM products", engine)
df_sellers = pd.read_sql("SELECT * FROM sellers", engine)
df_geolocation = pd.read_sql("SELECT * FROM geolocation", engine)
df_category_translation = pd.read_sql("SELECT * FROM product_category_name_translation", engine)

print("✅")

✅


In [4]:
# Basic checks for all 9 tables
tables = {
    "orders": df_orders,
    "customers": df_customers,
    "order_items": df_order_items,
    "payments": df_payments,
    "reviews": df_reviews,
    "products": df_products,
    "sellers": df_sellers,
    "geolocation": df_geolocation,
    "category_translation": df_category_translation,
}

for name, df in tables.items():
    print(f"{name}: shape={df.shape}, full_row_duplicates={df.duplicated().sum()}")


orders: shape=(99441, 8), full_row_duplicates=0
customers: shape=(99441, 5), full_row_duplicates=0
order_items: shape=(112650, 7), full_row_duplicates=0
payments: shape=(103886, 5), full_row_duplicates=0
reviews: shape=(98410, 7), full_row_duplicates=0
products: shape=(32951, 9), full_row_duplicates=0
sellers: shape=(3095, 4), full_row_duplicates=0
geolocation: shape=(1000163, 5), full_row_duplicates=261831
category_translation: shape=(71, 2), full_row_duplicates=0


 **ليش ما بنضيفهم في Notebook 1؟**

* هدفنا نعمل جدول واحد، **كل صف فيه يمثل `order_id` واحد**.
* **Geolocation:** ما فيه `order_id` أو رابط مباشر بالطلبية، وربطه بده أكثر من خطوة، فبنخليه لمرحلة **Feature Engineering** لاحقًا.
* **Category Translation:** هذا بس جدول ترجمة لأسماء الـ categories، وما فيه معلومات عن الطلبيات، فمش ضروري حاليًا.
* **الخلاصة:** بنفحصهم عشان نفهم الـ structure والـ duplicates، بس **ما بنعمل لهم merge في Notebook 1** لأنهم مش من الـ **core structure** للـ order-level table.


In [5]:
# Check if order_id repeats (multiple rows per order)
print("unique orders:", df_orders["order_id"].nunique())
print("order_items rows:", len(df_order_items), "| unique order_id:", df_order_items["order_id"].nunique())
print("payments rows:", len(df_payments), "| unique order_id:", df_payments["order_id"].nunique())

unique orders: 99441
order_items rows: 112650 | unique order_id: 98666
payments rows: 103886 | unique order_id: 99440


**ليش بنعمل هالفحص؟**


قبل ما نعمل **merge**، بدنا نتأكد إذا كل `order_id` إله صف واحد أو أكثر.

* إذا `len(df) == df["order_id"].nunique()` → كل طلبية إلها صف واحد، فبنقدر نعمل **merge مباشرة**.
* إذا `len(df) > df["order_id"].nunique()` → فيه أكثر من صف لنفس الطلبية، فلازم نعمل **aggregation** قبل الدمج.

### النتائج

* **orders:** فيه 99,441 طلبية، وهذا هو العدد المرجعي.
* **order_items:** فيه 112,650 صف، لكن بس 98,666 `order_id` فريدة → يعني بعض الطلبيات فيها أكثر من منتج، فلازم نعمل **aggregation**. كمان فيه 775 طلبية ما إلها `order_items`.
* **payments:** فيه 103,886 صف، لكن 99,440 `order_id` فريدة → يعني بعض الطلبيات إلها أكثر من عملية دفع، فلازم نعمل **aggregation**. وفيه طلبية وحدة ما إلها سجل دفع.

### الخلاصة

ما بنقدر نعمل merge لـ `order_items` و`payments` مباشرة، لأنه رح يكرر الطلبيات.

لذلك أول إشي بنعمل **aggregation**، وبعدها بنعمل merge، عشان بالنهاية يضل عندنا:

> **صف واحد لكل `order_id`.**


In [6]:
items_agg = df_order_items.groupby("order_id").agg(
    n_items=("order_item_id", "count"),
    total_price=("price", "sum"),
    total_freight=("freight_value", "sum")
).reset_index()

print(items_agg.shape)
items_agg.head()

(98666, 4)


,order_id,n_items,total_price,total_freight
0,00010242fe8c5a6d1ba2dd792cb16214,1,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,1,199.00,17.87
3,00024acbcdf0a6daa1e931b038114c75,1,12.99,12.79
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,199.90,18.14


In [7]:
payments_agg = df_payments.groupby("order_id").agg(
    total_payment=("payment_value", "sum"),
    n_payments=("payment_sequential", "count")
).reset_index()

print(payments_agg.shape)
payments_agg.head()

(99440, 3)


,order_id,total_payment,n_payments
0,00010242fe8c5a6d1ba2dd792cb16214,72.19,1
1,00018f77f2f0320c557190d7a144bdd3,259.83,1
2,000229ec398224ef6ca0657da4fc703e,216.87,1
3,00024acbcdf0a6daa1e931b038114c75,25.78,1
4,00042b26cf59d7ce69dfabb4e55b4fd9,218.04,1


In [8]:
df_ml = df_orders \
    .merge(items_agg, on="order_id", how="left") \
    .merge(payments_agg, on="order_id", how="left") \
    .merge(df_customers, on="customer_id", how="left")

print(df_ml.shape)
df_ml.head()

(99441, 17)


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,n_items,total_price,total_freight,total_payment,n_payments,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,1.0,29.99,8.72,38.71,3.0,7c396fd4830fd04220f754e42b4e5bff,3149,sao paulo,SP
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,1.0,118.70,22.76,141.46,1.0,af07308b275d755c9edb36a90c618231,47813,barreiras,BA
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,1.0,159.90,19.22,179.12,1.0,3a653a41f6f9fc3d2a113cf8398680e8,75265,vianopolis,GO
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,1.0,45.00,27.20,72.20,1.0,7c142cf63193a1473d2e66489a9ae977,59296,sao goncalo do amarante,RN
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,1.0,19.90,8.72,28.62,1.0,72632f0f9dd73dfee390c9b22eb56dd6,9195,santo andre,SP


In [9]:
assert df_ml["order_id"].nunique() == len(df_ml), "Duplicates found! Check the merge"
print("final row count:", len(df_ml))
print("unique order_id:", df_ml["order_id"].nunique())

final row count: 99441
unique order_id: 99441


In [10]:
import os
os.makedirs("data/processed", exist_ok=True)

df_ml.to_parquet("data/processed/notebook1_ml_table.parquet", index=False)
print("Saved ✅")

Saved ✅


In [11]:
check = pd.read_parquet("data/processed/notebook1_ml_table.parquet")
print(check.shape)
check.tail()

(99441, 17)


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,n_items,total_price,total_freight,total_payment,n_payments,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
99436,9c5dedf39a927c1b2549525ed64a053c,39bd1228ee8140590ac3aca26f2dfe00,delivered,2017-03-09 09:54:05,2017-03-09 09:54:05,2017-03-10 11:18:03,2017-03-17 15:08:01,2017-03-28,1.0,72.00,13.08,85.08,1.0,6359f309b166b0196dbf7ad2ac62bb5a,12209,sao jose dos campos,SP
99437,63943bddc261676b46f01ca7ac2f7bd8,1fca14ff2861355f6e5f14306ff977a7,delivered,2018-02-06 12:58:58,2018-02-06 13:10:37,2018-02-07 23:22:42,2018-02-28 17:37:56,2018-03-02,1.0,174.90,20.10,195.00,1.0,da62f9e57a76d978d02ab5362c509660,11722,praia grande,SP
99438,83c1379a015df1e13d02aae0204711ab,1aa71eb042121263aafbe80c1b562c9c,delivered,2017-08-27 14:46:43,2017-08-27 15:04:16,2017-08-28 20:52:26,2017-09-21 11:24:17,2017-09-27,1.0,205.99,65.02,271.01,1.0,737520a9aad80b3fbbdad19b66b37b30,45920,nova vicosa,BA
99439,11c177c8e97725db2631073c19f07b62,b331b74b18dc79bcdf6532d51e1637c1,delivered,2018-01-08 21:28:27,2018-01-08 21:36:21,2018-01-12 15:35:03,2018-01-25 23:32:54,2018-02-15,2.0,359.98,81.18,441.16,1.0,5097a5312c8b157bb7be58ae360ef43c,28685,japuiba,RJ
99440,66dea50a8b16d9b4dee7af250b4be1a5,edb027a75a1449115f6b43211ae02a24,delivered,2018-03-08 20:57:30,2018-03-09 11:20:28,2018-03-09 22:11:59,2018-03-16 13:08:30,2018-04-03,1.0,68.50,18.36,86.86,1.0,60350aa974b26ff12caad89e55993bd6,83750,lapa,PR
